# 🧠 Analyzing Mental Health Trends in the Workplace Using Big Data Analytics
**Author:** Vaibhav Mistry  
**Technologies:** Apache Spark, Spark SQL, Spark MLlib, Apache Hive, Hadoop HDFS  
**Dataset:** Mental Health in Tech Survey (1,260 records, 24 attributes)

---
## Executive Overview
Workplace mental health has emerged as a paramount determinant of organizational performance and employee retention. This notebook demonstrates a complete distributed data engineering and machine learning pipeline to analyze risk factors, normalize multi-source survey data, and build predictive classifiers for early intervention.

In [1]:
# Initialize Local SparkSession for Distributed Analytics
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, count, sum as spark_sum
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder \n    .appName("Workplace_Mental_Health_BigData") \n    .master("local[*]") \n    .getOrCreate()

print("Spark Version:", spark.version)

## 1. Data Ingestion & Quality Audit
Loading the dataset using schema inference and header parsing, followed by a distributed null audit.

In [2]:
data_path = "../data/Mental_Health_in_Tech_Survey_dataset.csv"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(data_path)
print(f"Total Records: {df.count()}, Total Attributes: {len(df.columns)}")
df.printSchema()
df.createOrReplaceTempView("mental_health_survey")

## 2. Distributed Exploratory Data Analysis & Normalization (Spark SQL)
### 2.1 Demographic Normalization (Free-Text Gender)

In [3]:
gender_df = spark.sql("""
  SELECT 
    CASE 
      WHEN LOWER(Gender) IN ('male', 'm') THEN 'Male'
      WHEN LOWER(Gender) IN ('female', 'f') THEN 'Female'
      ELSE 'Others'
    END AS Gender_Normalized,
    COUNT(*) AS Count
  FROM mental_health_survey
  GROUP BY 
    CASE 
      WHEN LOWER(Gender) IN ('male', 'm') THEN 'Male'
      WHEN LOWER(Gender) IN ('female', 'f') THEN 'Female'
      ELSE 'Others'
    END
""")
gender_df.show()

### 2.2 Workplace Factors: Work Interference vs Supervisor Support

In [4]:
factors_df = spark.sql("""
  SELECT work_interfere, supervisor, COUNT(*) AS Count
  FROM mental_health_survey
  WHERE treatment = 'Yes'
  GROUP BY work_interfere, supervisor
  ORDER BY Count DESC
""")
factors_df.show(10)

### 2.3 Company Size Normalization (Resolving Excel Date Auto-Formatting)

In [5]:
company_size_df = spark.sql("""
  SELECT 
    supervisor,
    CASE 
      WHEN no_employees = 'Jun-25' THEN '1-25'
      WHEN no_employees = '01-May' THEN '1-5'
      ELSE no_employees
    END AS normalized_no_employees,
    COUNT(*) AS Count
  FROM mental_health_survey
  WHERE treatment = 'Yes'
  GROUP BY supervisor,
    CASE 
      WHEN no_employees = 'Jun-25' THEN '1-25'
      WHEN no_employees = '01-May' THEN '1-5'
      ELSE no_employees
    END
  ORDER BY Count DESC
""")
company_size_df.show(10)

## 3. Spark MLlib Machine Learning Pipeline
Predicting treatment-seeking probability using indexed demographic and workplace attributes.

In [6]:
# Index categorical variables
idx_fam = StringIndexer(inputCol="family_history", outputCol="family_history_index", handleInvalid="skip")
idx_work = StringIndexer(inputCol="work_interfere", outputCol="work_interfere_index", handleInvalid="skip")
idx_treat = StringIndexer(inputCol="treatment", outputCol="label", handleInvalid="skip")

df_indexed = idx_fam.fit(df).transform(df)
df_indexed = idx_work.fit(df_indexed).transform(df_indexed)
df_indexed = idx_treat.fit(df_indexed).transform(df_indexed)

# Assemble features
assembler = VectorAssembler(inputCols=["Age", "family_history_index", "work_interfere_index"], outputCol="features")
final_data = assembler.transform(df_indexed.filter(df_indexed["Age"].between(18, 75)))

# 80/20 Train Test Split
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)
print(f"Train size: {train_data.count()}, Test size: {test_data.count()}")

### 3.1 Logistic Regression Model & AUC Evaluation

In [7]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)
lr_preds = lr_model.transform(test_data)

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
lr_auc = evaluator.evaluate(lr_preds)
print(f"Logistic Regression Area Under ROC: {lr_auc:.4f}")

# Feature coefficients
weights = lr_model.coefficients.toArray()
feat_names = ["Age", "family_history", "work_interfere"]
for name, w in zip(feat_names, weights):
    print(f"Feature {name}: coefficient = {w:.6f}")

### 3.2 Decision Tree Classifier

In [8]:
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
dt_model = dt.fit(train_data)
dt_preds = dt_model.transform(test_data)
dt_auc = evaluator.evaluate(dt_preds)
print(f"Decision Tree Area Under ROC: {dt_auc:.4f}")

## 4. Key Findings & Recommendations
1. **Workplace Interference is the Primary Driver**: Employees experiencing work interference from mental health symptoms are significantly more likely to require treatment.
2. **Wellness Program Deficit**: Over 60% of respondents report having no access to mental health wellness programs, indicating a major corporate gap.
3. **Manager Training is Essential**: 39% of treatment seekers cite supportive supervisors, underscoring the need for frontline leadership mental health education.